# 03d · RAG report: tables, figures and judge check

Reads `results/rag/retrieval.json` and `generation.json` (no model calls). If the team has filled in
`03_rag/data/judge_check.csv`, it also reports how often the LLM judge agrees with people on faithfulness.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common import config, env

config.ensure_dirs()
ret = json.loads((config.RAG_RESULTS / "retrieval.json").read_text())
gen = json.loads((config.RAG_RESULTS / "generation.json").read_text())
ABL = ["Dense only", "BM25 only", "Dense+BM25 (RRF)", "RRF + rerank", "Full PlantIQ (+topic, MMR)"]
SK = "Selva Kumar et al. (dense, LangChain)"

## Retrieval

In [ ]:
rs = pd.DataFrame(ret["summary"]).T
rs[["recall@1", "recall@3", "recall@5", "recall@10", "hit@3", "ndcg@3", "mrr@10"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.2))
labels = [SK] + ABL
short = ["Selva Kumar\n(dense)", "Dense", "BM25", "Dense+BM25\n(RRF)", "RRF+\nrerank", "Full\n(+topic, MMR)"]
x = np.arange(len(labels))
for off, m, c in ((-0.18, "recall@3", "#4c78a8"), (0.18, "ndcg@3", "#f58518")):
    vals = [ret["summary"][s][m] for s in labels]
    ax.bar(x + off, vals, 0.36, label=m.replace("@", "@"), color=c)
ax.set_xticks(x, short, fontsize=8)
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.set_title(f"Retrieval on {ret['n_questions']} questions (gold-quote relevance)", fontsize=9)
plt.tight_layout()
for ext in ("png", "pdf"):
    plt.savefig(config.FIG_DIR / f"fig_rag_ablation.{ext}", dpi=200)
plt.show()

## Generation

In [ ]:
S = gen["systems"]
M_ = ["faithfulness", "answer_relevancy", "context_precision", "answer_correctness"]
gt = pd.DataFrame({v["name"]: {**{m: v[m]["mean"] for m in M_},
                                "unsupported doses": v["unsupported_dose_rate"],
                                "abstain out-of-KB": v["abstention_out_of_kb"],
                                "false abstain in-KB": v["false_abstention_in_kb"],
                                "latency p50 (s)": v["latency_median_s"], "latency p95 (s)": v["latency_p95_s"]}
                   for v in S.values()}).T
gt

In [ ]:
order = [o for o in ("PQ", "SK", "L0") if o in S]
colors = {"PQ": "#e6550d", "SK": "#6baed6", "L0": "#bdbdbd"}
fig, ax = plt.subplots(figsize=(7.2, 4.0))
w = 0.8 / len(order)
for i, s in enumerate(order):
    vals = [S[s][m]["mean"] for m in M_]
    err = np.array([[S[s][m]["mean"] - S[s][m]["lo"], S[s][m]["hi"] - S[s][m]["mean"]] for m in M_]).T
    err = np.nan_to_num(err)
    ax.bar(np.arange(len(M_)) + (i - (len(order) - 1) / 2) * w, np.nan_to_num(vals), w, yerr=err, capsize=2, label=S[s]["name"],
           color=colors[s])
ax.set_xticks(range(len(M_)), ["Faithfulness", "Answer relevancy", "Context precision", "Answer correctness"],
              fontsize=8)
ax.set_ylim(0, 1)
ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False)
ax.set_title(f"Generation quality (judge: {gen['models']['judge']}; 95% bootstrap CI)", fontsize=9)
plt.tight_layout()
for ext in ("png", "pdf"):
    plt.savefig(config.FIG_DIR / f"fig_generation.{ext}", dpi=200)
plt.show()

## Judge vs people (optional, after the team fills `judge_check.csv`)

In [ ]:
chk_path = config.RAG_DATA / "judge_check.csv"
agreement = None
if chk_path.exists():
    chk = pd.read_csv(chk_path, encoding="utf-8-sig")
    cols = [c for c in chk.columns if c.startswith("human_faithful")]
    filled = chk.dropna(subset=cols)
    if len(filled):
        human = filled[cols].astype(float).mean(1)
        agreement = {"n": int(len(filled)),
                     "mean_abs_diff": float((human - filled["faithfulness"]).abs().mean()),
                     "pearson": float(np.corrcoef(human, filled["faithfulness"])[0, 1]) if len(filled) > 2 else None,
                     "human_A_vs_B_mean_abs_diff": float((filled[cols[0]].astype(float) - filled[cols[1]].astype(float)).abs().mean())}
print("judge agreement:", agreement or "not filled in yet")

## Save paper numbers (same schema as IEEE-Paper/scripts/mock_data.json)

In [ ]:
paper = {
    "rag_ablation": {"is_mock": False, "configs": ["Dense only", "BM25 only", "Dense+BM25 (RRF)", "RRF + rerank",
                                                   "Full (+topic, MMR)"],
                     "Recall@3": [round(ret["summary"][c]["recall@3"], 4) for c in ABL],
                     "nDCG@3": [round(ret["summary"][c]["ndcg@3"], 4) for c in ABL]},
    "generation_comparison": {"is_mock": False, "metrics": ["Faithfulness", "Answer relevance", "Context precision"],
                              "systems": {S[s]["name"]: [round(S[s][m]["mean"], 4) if S[s][m]["mean"] == S[s][m]["mean"]
                                                         else 0.0 for m in ("faithfulness", "answer_relevancy",
                                                                            "context_precision")]
                                          for s in ("PQ", "SK", "L0") if s in S}},
    "table_viii": [{"system": S[s]["name"],
                    "recall@3": (round(ret["summary"]["Full PlantIQ (+topic, MMR)" if s == "PQ" else SK]["recall@3"], 3)
                                 if s in ("PQ", "SK") else None),
                    "faithfulness": round(S[s]["faithfulness"]["mean"], 3),
                    "unsupported_doses_pct": round(100 * S[s]["unsupported_dose_rate"], 1),
                    "latency_s": round(S[s]["latency_median_s"], 1)} for s in ("L0", "SK", "PQ") if s in S],
    "judge_agreement": agreement,
}
env.write_json(config.RAG_RESULTS / "paper_rag.json", paper)
md = ["## Module 2 (RAG)", "", f"{ret['n_questions']} in-KB questions; generator {gen['models']['generator']}, "
      f"baseline {gen['models']['baseline']}, judge {gen['models']['judge']}.", "", "### Retrieval", "",
      rs.to_markdown(floatfmt=".3f"), "", "### Generation", "", gt.to_markdown(floatfmt=".3f"), ""]
if agreement:
    md += [f"Judge vs people (faithfulness, n={agreement['n']}): mean |diff| {agreement['mean_abs_diff']:.3f}", ""]
(config.RESULTS_DIR / "RESULTS_rag.md").write_text("\n".join(md), encoding="utf-8")
print("\n".join(md))